# Задача 2. Permutations

**Объяснение решения**

Используем перебор с возвратом. В `path` собираем текущую перестановку,
а в `used` отмечаем, какие элементы массива уже взяли.

На каждом шаге:

1. Если длина `path` равна длине массива, добавляем копию `path` в результат
2. Иначе выбираем очередной свободный элемент, отмечаем его в `used`
   и добавляем в `path`
3. Рекурсивно продолжаем перестановку
4. После возврата убираем последний элемент из `path` и снимаем отметку
   в `used`

Для пустого массива возвращаем `[[]]`: у него одна перестановка - пустая.

**Решение**

Декоратор беру из первой задачи

In [1]:
from contextvars import ContextVar
from functools import wraps


_depth = ContextVar("tracer_depth", default=0)


def _format_arguments(args, kwargs):
    arguments = [repr(value) for value in args]
    arguments.extend(f"{name}={value!r}" for name, value in kwargs.items())
    return ", ".join(arguments)


def tracer(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        depth = _depth.get()
        print(f"{'  ' * depth}-> {function.__name__}({_format_arguments(args, kwargs)})")
        token = _depth.set(depth + 1)

        try:
            result = function(*args, **kwargs)
        except Exception as error:
            print(f"{'  ' * depth}!! {function.__name__}: {error!r}")
            raise
        else:
            print(f"{'  ' * depth}<- {function.__name__} = {result!r}")
            return result
        finally:
            _depth.reset(token)

    return wrapper

In [2]:
def permute(nums: list[int]) -> list[list[int]]:
    values = sorted(nums)
    result = []
    used = [False] * len(values)

    @tracer
    def backtrack(path):
        if len(path) == len(values):
            result.append(path.copy())
            return 1

        count = 0

        for i, value in enumerate(values):
            if used[i]:
                continue
            if i > 0 and value == values[i - 1] and not used[i - 1]:
                continue

            used[i] = True
            path.append(value)

            count += backtrack(path)

            path.pop()
            used[i] = False

        return count

    backtrack([])
    return result

**Пример**

`nums = [1, 2, 3]`. При каждом рекурсивном вызове в `path` появляется ещё одно число. У готовой перестановки длина равна `3`.

В выводе `->` показывает вход в рекурсию, `<-` - возврат. К Лист возвращает `1`, а корень - `6`.

In [3]:
nums = [1, 2, 3]
result = permute(nums)
print("Перестановки:", result)

-> backtrack([])
  -> backtrack([1])
    -> backtrack([1, 2])
      -> backtrack([1, 2, 3])
      <- backtrack = 1
    <- backtrack = 1
    -> backtrack([1, 3])
      -> backtrack([1, 3, 2])
      <- backtrack = 1
    <- backtrack = 1
  <- backtrack = 2
  -> backtrack([2])
    -> backtrack([2, 1])
      -> backtrack([2, 1, 3])
      <- backtrack = 1
    <- backtrack = 1
    -> backtrack([2, 3])
      -> backtrack([2, 3, 1])
      <- backtrack = 1
    <- backtrack = 1
  <- backtrack = 2
  -> backtrack([3])
    -> backtrack([3, 1])
      -> backtrack([3, 1, 2])
      <- backtrack = 1
    <- backtrack = 1
    -> backtrack([3, 2])
      -> backtrack([3, 2, 1])
      <- backtrack = 1
    <- backtrack = 1
  <- backtrack = 2
<- backtrack = 6
Перестановки: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


**Пошаговая работа `backtrack`.** Сначала перебираем все перестановки, которые начинаются с `1`, затем с `2` и с `3`.

| Номер | Цепочка вызовов: значение `path` | Добавляем в результат |
|---|---|---|
| 1 | `[] → [1] → [1, 2] → [1, 2, 3]` | `[1, 2, 3]` |
| 2 | `[] → [1] → [1, 3] → [1, 3, 2]` | `[1, 3, 2]` |
| 3 | `[] → [2] → [2, 1] → [2, 1, 3]` | `[2, 1, 3]` |
| 4 | `[] → [2] → [2, 3] → [2, 3, 1]` | `[2, 3, 1]` |
| 5 | `[] → [3] → [3, 1] → [3, 1, 2]` | `[3, 1, 2]` |
| 6 | `[] → [3] → [3, 2] → [3, 2, 1]` | `[3, 2, 1]` |

Например, после добавления `[1, 2, 3]` убираем `3`, затем возвращаемся к `path = [1]` и убираем `2`. На следующей ветке выбираем `3`, потом `2`. Получаем `[1, 3, 2]`.

**Сложность**

Пусть `n` - длина массива. В худшем случае все элементы различны, и число перестановок равно `n!`.

| Что учитываем | Время | Память |
|---|---|---|
| Алгоритм без печати трассировки | `O(n · n!)` в худшем случае | `O(n)` дополнительно |
| Список всех перестановок | — | `O(n · n!)` в худшем случае |

В каждой готовой перестановке `n` элементов, и для сохранения нужна копия длины `n`. Кроме этого, на каждом неполном пути перебираем индексы массива.
Вместе это даёт `O(n · n!)`.

В `path`, `used` и копии массива хранится не больше `n` элементов. Доп память - `O(n)`.

**Тесты**

In [4]:
import math
import unittest
from contextlib import redirect_stdout
from io import StringIO
from itertools import permutations as reference_permutations


class TestPermutations(unittest.TestCase):
    def capture(self, nums):
        output = StringIO()
        with redirect_stdout(output):
            result = permute(nums)
        return result, output.getvalue().splitlines()

    def test_three_elements(self):
        result, _ = self.capture([1, 2, 3])
        self.assertEqual(result, [
            [1, 2, 3], [1, 3, 2],
            [2, 1, 3], [2, 3, 1],
            [3, 1, 2], [3, 2, 1],
        ])

    def test_two_elements(self):
        result, _ = self.capture([0, 1])
        self.assertEqual(result, [[0, 1], [1, 0]])

    def test_single_element(self):
        result, _ = self.capture([1])
        self.assertEqual(result, [[1]])

    def test_empty(self):
        result, lines = self.capture([])
        self.assertEqual(result, [[]])
        self.assertEqual(lines, [
            "-> backtrack([])",
            "<- backtrack = 1",
        ])

    def test_duplicates(self):
        result, _ = self.capture([1, 1, 2])
        self.assertEqual(result, [[1, 1, 2], [1, 2, 1], [2, 1, 1]])

    def test_two_duplicate_groups(self):
        result, _ = self.capture([2, 1, 2, 1])
        self.assertEqual(result, [
            [1, 1, 2, 2],
            [1, 2, 1, 2],
            [1, 2, 2, 1],
            [2, 1, 1, 2],
            [2, 1, 2, 1],
            [2, 2, 1, 1],
        ])

    def test_all_equal(self):
        result, _ = self.capture([5, 5, 5])
        self.assertEqual(result, [[5, 5, 5]])

    def test_unsorted_input(self):
        result, _ = self.capture([3, 1, 2])
        self.assertEqual(result, [
            [1, 2, 3], [1, 3, 2],
            [2, 1, 3], [2, 3, 1],
            [3, 1, 2], [3, 2, 1],
        ])

    def test_negative_and_zero(self):
        result, _ = self.capture([0, -1, 2])
        self.assertEqual(result, [
            [-1, 0, 2], [-1, 2, 0],
            [0, -1, 2], [0, 2, -1],
            [2, -1, 0], [2, 0, -1],
        ])

    def test_input_unchanged(self):
        nums = [3, 1, 2, 1]
        before = nums.copy()
        self.capture(nums)
        self.assertEqual(nums, before)

    def test_independent_results(self):
        result, _ = self.capture([1, 2, 3])
        self.assertEqual(len({id(item) for item in result}), 6)
        rest = [item.copy() for item in result[1:]]
        result[0].append(99)
        self.assertEqual(result[1:], rest)

    def test_repeated_calls(self):
        first, _ = self.capture([0, 1])
        first[0][0] = 99
        second, lines = self.capture([0, 1])
        self.assertEqual(second, [[0, 1], [1, 0]])
        self.assertEqual(lines[0], "-> backtrack([])")
        self.assertEqual(lines[-1], "<- backtrack = 2")

    def test_count_and_uniqueness(self):
        nums = [0, 1, 2, 3, 4, 5]
        result, _ = self.capture(nums)
        self.assertEqual(len(result), math.factorial(len(nums)))
        self.assertEqual(len({tuple(item) for item in result}), len(result))
        for item in result:
            self.assertEqual(sorted(item), nums)

    def test_against_reference(self):
        cases = [
            [], [0], [2, 1], [3, -1, 0],
            [4, 1, 3, 2], [2, 2, 1], [0, -1, 0, -1],
        ]
        for nums in cases:
            with self.subTest(nums=nums):
                result, _ = self.capture(nums)
                expected = sorted(set(reference_permutations(nums)))
                self.assertEqual(result, [list(item) for item in expected])

    def test_trace(self):
        result, lines = self.capture([0, 1])
        self.assertEqual(result, [[0, 1], [1, 0]])
        self.assertEqual(lines, [
            "-> backtrack([])",
            "  -> backtrack([0])",
            "    -> backtrack([0, 1])",
            "    <- backtrack = 1",
            "  <- backtrack = 1",
            "  -> backtrack([1])",
            "    -> backtrack([1, 0])",
            "    <- backtrack = 1",
            "  <- backtrack = 1",
            "<- backtrack = 2",
        ])

In [5]:
suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestPermutations)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful(), "Некоторые тесты завершились ошибкой"
print(f"Успешно пройдено тестов: {test_result.testsRun}")

test_against_reference (__main__.TestPermutations.test_against_reference) ... ok
test_all_equal (__main__.TestPermutations.test_all_equal) ... ok
test_count_and_uniqueness (__main__.TestPermutations.test_count_and_uniqueness) ... ok
test_duplicates (__main__.TestPermutations.test_duplicates) ... ok
test_empty (__main__.TestPermutations.test_empty) ... ok
test_independent_results (__main__.TestPermutations.test_independent_results) ... ok
test_input_unchanged (__main__.TestPermutations.test_input_unchanged) ... ok
test_negative_and_zero (__main__.TestPermutations.test_negative_and_zero) ... ok
test_repeated_calls (__main__.TestPermutations.test_repeated_calls) ... ok
test_single_element (__main__.TestPermutations.test_single_element) ... ok
test_three_elements (__main__.TestPermutations.test_three_elements) ... ok
test_trace (__main__.TestPermutations.test_trace) ... ok
test_two_duplicate_groups (__main__.TestPermutations.test_two_duplicate_groups) ... ok
test_two_elements (__main__.Tes

Успешно пройдено тестов: 15
